# Question 1 – Interactive Web Map of Waste-Bin Observations

This notebook turns the waste-bin observations collected in the field with **QField** into an interactive **Folium** web map.

**What the map provides**
- every waste bin as a clickable point, coloured by observed usage level (Low / Medium / High);
- a **tooltip** on hover (bin name, usage, collector) and a **popup** on click with the title, field note, date/time, coordinates, the **photograph** taken in the field, and the **name of the student who collected it**;
- a **Layer Control** to switch base maps (street map, light map, satellite) and toggle each usage-level layer;
- full-screen, mini-map, distance/area measurement, "locate me", live cursor coordinates and a scale bar;
- a legend and a data-credit panel naming every contributing student.

**Hosting.** Photos are resized and embedded directly inside the HTML (base64), so the single file `web_map/index.html` works on any web host (GitHub Pages, Google Sites embed, personal server). The photos do not depend on files on the local computer.

**Data from other students.** To add a classmate's QField project, append one entry to `SOURCES` in the configuration cell with `own=False`. Their name then appears in the popup as the original collector, with an acknowledgement line, and in the data-credit panel.

In [1]:
import base64
import io
import math
from pathlib import Path

import pandas as pd
import geopandas as gpd
import folium
from folium import plugins
from branca.element import Element, MacroElement, Template
from PIL import Image, ImageOps

## 1. Configuration
Paths are relative to this notebook's folder. Each entry in `SOURCES` is one QField project (one student).

In [2]:
BASE_DIR = Path(".").resolve()

MAP_AUTHOR = "Susovan (CS25M258)"          # student who prepared / submits this map
MAP_TITLE  = "Waste-Bin Observations — Urban Field Survey (QField)"

SOURCES = [
    dict(
        collector="Susovan",                  # student who collected these observations
        roll_no="CS25M258",
        project_dir=BASE_DIR / "Urban_project_susovan",
        gpkg="notes.gpkg",
        point_layer="notes_point",
        attachment_layer="notes_attachments",
        own=True,                             # False => data from another student (acknowledged in popup)
    ),
    # Example for a classmate's data (uncomment and edit):
    # dict(collector="Classmate Name", roll_no="CS25Mxxx",
    #      project_dir=BASE_DIR / "Urban_project_classmate", gpkg="notes.gpkg",
    #      point_layer="notes_point", attachment_layer="notes_attachments", own=False),
]

OUT_DIR = BASE_DIR / "web_map"
OUT_HTML = OUT_DIR / "index.html"            # index.html => served at the folder URL on GitHub Pages

PHOTO_MAX_PX = 720         # longest side of embedded photo
PHOTO_QUALITY = 68         # JPEG quality of embedded photo
OUTLIER_KM = 10            # drop points further than this from the survey's median location (GPS test fixes)
LOCAL_TZ = "Asia/Kolkata"

USAGE_ORDER  = ["Low usage", "Medium usage", "High usage"]
USAGE_COLORS = {"Low usage": "#1a9850", "Medium usage": "#f39c12", "High usage": "#d73027"}

## 2. Load observations and link each one to its photograph
QField stores each note as a point in `notes_point` and its photo in `notes_attachments`, linked by the note's `uuid`.

In [3]:
def load_source(src):
    gpkg = src["project_dir"] / src["gpkg"]
    pts = gpd.read_file(gpkg, layer=src["point_layer"])
    att = pd.DataFrame(gpd.read_file(gpkg, layer=src["attachment_layer"]))

    pts["lon"] = pts.geometry.x
    pts["lat"] = pts.geometry.y
    df = pd.DataFrame(pts.drop(columns="geometry"))

    # one photo per note (first attachment if several)
    att = att.dropna(subset=["media"]).drop_duplicates("note_uuid")[["note_uuid", "media"]]
    df = df.merge(att, how="left", left_on="uuid", right_on="note_uuid").drop(columns="note_uuid")

    df["photo_path"] = df["media"].map(lambda m: src["project_dir"] / m if isinstance(m, str) else None)
    df["collector"] = src["collector"]
    df["roll_no"] = src["roll_no"]
    df["own"] = src["own"]
    df["source_project"] = src["project_dir"].name
    return df

raw = pd.concat([load_source(s) for s in SOURCES], ignore_index=True)
print("Observations loaded:", len(raw))
print("With a linked photo:", raw["photo_path"].notna().sum())
raw[["title", "note", "timestamp", "lat", "lon", "media", "collector"]].head(10)

Observations loaded: 39
With a linked photo: 37


        title          note                 timestamp        lat        lon                            media collector
0   Dustbin 1     Low usage  2026-09-01T12:48:09.367Z  23.294544  81.664963  DCIM/JPEG_20260901124943855.jpg   Susovan
1   Dustbin 2     Low usage  2026-09-01T12:51:16.849Z  23.294544  80.734259  DCIM/JPEG_20260901125140530.jpg   Susovan
2   Dustbin 3     Low usage  2026-09-01T12:54:19.399Z  13.713141  79.592066  DCIM/JPEG_20260901125434456.jpg   Susovan
3   Dustbin 4     Low usage  2026-09-01T13:04:22.079Z  13.719996  79.590160  DCIM/JPEG_20260901130453455.jpg   Susovan
4   Dustbin 5  Medium usage  2026-09-01T13:06:42.352Z  13.714036  79.592959  DCIM/JPEG_20260901130655840.jpg   Susovan
5   Dustbin 7    High usage  2026-09-03T12:00:54.052Z  13.716186  79.593970  DCIM/JPEG_20260903120113903.jpg   Susovan
6   Dustbin 6    High usage  2026-09-03T12:03:50.983Z  13.715011  79.590531  DCIM/JPEG_20260903120409127.jpg   Susovan
7   Dustbin 8     Low usage  2026-09-03T12:27:07

## 3. Clean the observations
- remove empty notes (no title and no field note);
- standardise the usage label (`Moderate usage` → `Medium usage`, strip stray newlines);
- convert the UTC timestamp to local time (IST);
- remove GPS outliers: two early test points were recorded hundreds of km away from the survey area.

In [4]:
df = raw.copy()

empty = df["title"].isna() & df["note"].isna()
print("Empty notes removed:", int(empty.sum()))
df = df.loc[~empty].copy()

df["title"] = df["title"].fillna("Untitled bin").str.strip()
df["usage"] = (df["note"].fillna("").str.strip()
                 .str.replace("Moderate usage", "Medium usage", regex=False))
df.loc[~df["usage"].isin(USAGE_ORDER), "usage"] = "Not recorded"

df["time_local"] = pd.to_datetime(df["timestamp"], utc=True, format="mixed").dt.tz_convert(LOCAL_TZ)

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi, dl = p2 - p1, math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * 6371.0 * math.asin(math.sqrt(a))

mlat, mlon = df["lat"].median(), df["lon"].median()
df["dist_km"] = [haversine_km(mlat, mlon, a, b) for a, b in zip(df["lat"], df["lon"])]
outliers = df[df["dist_km"] > OUTLIER_KM]
print(f"GPS outliers removed (> {OUTLIER_KM} km from survey centre):", len(outliers))
print(outliers[["title", "lat", "lon", "dist_km"]].round(3).to_string())
df = df[df["dist_km"] <= OUTLIER_KM].copy()

print("\nObservations kept:", len(df))
print(df["usage"].value_counts().reindex(USAGE_ORDER + ["Not recorded"]).dropna().astype(int).to_string())

Empty notes removed: 1
GPS outliers removed (> 10 km from survey centre): 2
       title     lat     lon   dist_km
0  Dustbin 1  23.295  81.665  1087.474
1  Dustbin 2  23.295  80.734  1072.120

Observations kept: 36
usage
Low usage       16
Medium usage     9
High usage      11


## 4. Separate points that share identical GPS coordinates
A few bins were saved before the GPS fix refreshed, so they have exactly the same coordinates and would sit on top of each other. They are spread on a ~4 m ring for display only; the popup still reports the original recorded coordinates and flags the issue.

In [5]:
df["disp_lat"], df["disp_lon"], df["shared_fix"] = df["lat"], df["lon"], False
for (_, _), grp in df.groupby([df["lat"].round(7), df["lon"].round(7)]):
    if len(grp) < 2:
        continue
    for k, idx in enumerate(grp.index):
        ang = 2 * math.pi * k / len(grp)
        r_m = 4.0
        df.at[idx, "disp_lat"] = df.at[idx, "lat"] + (r_m * math.cos(ang)) / 111_320
        df.at[idx, "disp_lon"] = df.at[idx, "lon"] + (r_m * math.sin(ang)) / (111_320 * math.cos(math.radians(df.at[idx, "lat"])))
        df.at[idx, "shared_fix"] = True
print("Points sharing an identical GPS fix:", int(df["shared_fix"].sum()))

Points sharing an identical GPS fix: 3


## 5. Prepare photographs for the web
Each photo is rotated using its EXIF orientation, resized, compressed and embedded in the HTML as base64. This keeps the map to one file and guarantees the photos load wherever the map is hosted.

In [6]:
def photo_to_data_uri(path):
    if path is None or not Path(path).exists():
        return None
    with Image.open(path) as im:
        im = ImageOps.exif_transpose(im).convert("RGB")
        im.thumbnail((PHOTO_MAX_PX, PHOTO_MAX_PX))
        buf = io.BytesIO()
        im.save(buf, format="JPEG", quality=PHOTO_QUALITY, optimize=True, progressive=True)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode("ascii")

df["photo_uri"] = df["photo_path"].map(photo_to_data_uri)
sizes_kb = df["photo_uri"].dropna().map(len) / 1024
print(f"Photos embedded: {df['photo_uri'].notna().sum()} / {len(df)}  "
      f"(avg {sizes_kb.mean():.0f} KB, total {sizes_kb.sum()/1024:.1f} MB)")

Photos embedded: 35 / 36  (avg 56 KB, total 1.9 MB)


## 6. Popup and tooltip content

In [7]:
def popup_html(r):
    color = USAGE_COLORS.get(r.usage, "#7f7f7f")
    photo = (f'<img src="{r.photo_uri}" alt="Photo of {r.title}" '
             f'style="width:100%;max-height:260px;object-fit:cover;border-radius:6px;margin:6px 0;">'
             if r.photo_uri else '<div style="padding:8px;color:#888;font-style:italic;">No photo recorded</div>')
    if r.own:
        credit = f"<b>Collected by:</b> {r.collector} ({r.roll_no})"
    else:
        credit = (f"<b>Collected by:</b> {r.collector} ({r.roll_no})<br>"
                  f"<span style='color:#a33;font-size:11px;'>Data courtesy of {r.collector} — "
                  f"used with acknowledgement</span>")
    gps_note = ("<div style='font-size:11px;color:#a66;margin-top:4px;'>Note: GPS position identical to "
                "another record; marker offset ~4 m for visibility.</div>" if r.shared_fix else "")
    return f'''
<div style="font-family:Segoe UI,Arial,sans-serif;font-size:12.5px;width:290px;line-height:1.45;">
  <div style="font-size:15px;font-weight:700;">{r.title}</div>
  <span style="display:inline-block;background:{color};color:#fff;padding:1px 8px;border-radius:10px;
        font-size:11px;font-weight:600;margin:2px 0 2px;">{r.usage}</span>
  {photo}
  <table style="width:100%;border-collapse:collapse;font-size:12px;">
    <tr><td style="color:#666;padding:1px 6px 1px 0;">Field note</td><td>{(r.note or '').strip() or '—'}</td></tr>
    <tr><td style="color:#666;padding:1px 6px 1px 0;">Recorded</td><td>{r.time_local:%d %b %Y, %H:%M} IST</td></tr>
    <tr><td style="color:#666;padding:1px 6px 1px 0;">Lat, Lon</td><td>{r.lat:.6f}, {r.lon:.6f}</td></tr>
    <tr><td style="color:#666;padding:1px 6px 1px 0;">Photo file</td><td style="word-break:break-all;">{r.media if isinstance(r.media, str) else '—'}</td></tr>
    <tr><td style="color:#666;padding:1px 6px 1px 0;">Source</td><td>QField project “{r.source_project}”</td></tr>
  </table>
  <div style="margin-top:6px;padding-top:5px;border-top:1px solid #ddd;">{credit}</div>
  {gps_note}
</div>'''

def tooltip_html(r):
    return f"<b>{r.title}</b> · {r.usage}<br><span style='color:#555'>Collected by {r.collector}</span>"

## 7. Build the interactive map

In [8]:
m = folium.Map(location=[df["lat"].mean(), df["lon"].mean()], zoom_start=16,
               tiles=None, control_scale=True)

# Base maps (radio buttons in the Layer Control)
folium.TileLayer("OpenStreetMap", name="Street map (OpenStreetMap)").add_to(m)
folium.TileLayer(
    tiles="https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png",
    attr="&copy; OpenStreetMap contributors &copy; CARTO", subdomains="abcd",
    name="Light map (CARTO)",
).add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri — Source: Esri, Maxar, Earthstar Geographics, GIS User Community",
    name="Satellite (Esri World Imagery)",
).add_to(m)

# One overlay layer per usage level (checkboxes in the Layer Control)
groups = {}
for u in USAGE_ORDER + ["Not recorded"]:
    n = int((df["usage"] == u).sum())
    if n:
        groups[u] = folium.FeatureGroup(name=f"{u} ({n})", show=True).add_to(m)

for r in df.itertuples():
    color = USAGE_COLORS.get(r.usage, "#7f7f7f")
    folium.CircleMarker(
        location=[r.disp_lat, r.disp_lon],
        radius=9, color="#222", weight=1.2,
        fill=True, fill_color=color, fill_opacity=0.9,
        tooltip=folium.Tooltip(tooltip_html(r), sticky=True),
        popup=folium.Popup(popup_html(r), max_width=320),
    ).add_to(groups[r.usage])

# Optional heat layer showing where bins are concentrated (off by default)
plugins.HeatMap(df[["lat", "lon"]].values.tolist(), name="Bin density (heatmap)",
                radius=28, blur=22, min_opacity=0.35, show=False).add_to(m)

# Map controls
plugins.Fullscreen(position="topleft").add_to(m)
plugins.LocateControl(position="topleft").add_to(m)
plugins.MeasureControl(position="topleft", primary_length_unit="meters",
                       primary_area_unit="sqmeters").add_to(m)
plugins.MousePosition(position="bottomright", separator=" | ", prefix="Lat | Lon:",
                      num_digits=6).add_to(m)
plugins.MiniMap(toggle_display=True, position="bottomleft", width=140, height=120).add_to(m)
folium.LayerControl(position="topright", collapsed=False).add_to(m)

m.fit_bounds([[df["lat"].min(), df["lon"].min()], [df["lat"].max(), df["lon"].max()]], padding=(30, 30))

## 8. Title, legend and data-credit panel

In [9]:
counts = df["usage"].value_counts()
legend_rows = "".join(
    f"<div><span style='display:inline-block;width:12px;height:12px;border-radius:50%;"
    f"background:{USAGE_COLORS[u]};border:1px solid #222;margin-right:6px;vertical-align:-1px;'></span>"
    f"{u} <span style='color:#777'>({int(counts.get(u, 0))})</span></div>"
    for u in USAGE_ORDER if counts.get(u, 0)
)

credit_rows = ""
for (name, roll, own), g in df.groupby(["collector", "roll_no", "own"]):
    tag = "" if own else " <i style='color:#a33'>(acknowledged)</i>"
    span = f"{g['time_local'].min():%d %b} – {g['time_local'].max():%d %b %Y}"
    credit_rows += f"<div>• <b>{name}</b> ({roll}) — {len(g)} bins, {span}{tag}</div>"

panel = f'''
{{% macro html(this, kwargs) %}}
<div id="map-title" style="position:fixed;top:12px;left:50%;transform:translateX(-50%);z-index:9999;
     background:rgba(255,255,255,0.94);padding:8px 16px;border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);
     font-family:Segoe UI,Arial,sans-serif;text-align:center;max-width:70vw;">
  <div style="font-size:16px;font-weight:700;">{MAP_TITLE}</div>
  <div style="font-size:11.5px;color:#555;">{len(df)} waste bins · hover for summary, click for photo &amp; details · map by {MAP_AUTHOR}</div>
</div>
<div id="map-legend" style="position:fixed;bottom:34px;right:12px;z-index:9999;background:rgba(255,255,255,0.95);
     padding:9px 12px;border-radius:8px;box-shadow:0 1px 6px rgba(0,0,0,.3);
     font-family:Segoe UI,Arial,sans-serif;font-size:12px;line-height:1.7;max-width:270px;">
  <div style="font-weight:700;margin-bottom:2px;">Observed bin usage</div>
  {legend_rows}
  <div style="font-weight:700;margin-top:6px;border-top:1px solid #ddd;padding-top:4px;">Data collected by</div>
  {credit_rows}
  <div style="color:#777;font-size:10.5px;margin-top:3px;">Collected with QField (mobile GIS)</div>
</div>
<style>
  @media (max-width: 640px) {{
    #map-title {{ top:auto; bottom:8px; max-width:92vw; padding:5px 10px; }}
    #map-title div:first-child {{ font-size:13px; }}
    #map-legend {{ bottom:auto; top:120px; right:8px; font-size:11px; max-width:190px; }}
  }}
  .leaflet-popup-content {{ margin:10px 12px; }}
</style>
{{% endmacro %}}
'''
overlay = MacroElement()
overlay._template = Template(panel)
m.get_root().add_child(overlay)
m.get_root().header.add_child(Element(f"<title>{MAP_TITLE}</title>"))

## 9. Save the HTML map
Upload the `web_map` folder (or just `index.html`) to your personal site. For GitHub Pages: push `index.html` to a repository, enable **Settings → Pages**, and the map is served at `https://<username>.github.io/<repo>/`.

In [10]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
m.save(str(OUT_HTML))
print("Saved:", OUT_HTML)
print(f"File size: {OUT_HTML.stat().st_size / 1024 / 1024:.2f} MB (photos embedded — no external image files needed)")

Saved: /home/claude/out/CS25M258-test-urban/web_map/index.html
File size: 2.03 MB (photos embedded — no external image files needed)


In [ ]:
m   # display the map inline in Jupyter